In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

MIMIC_DIR = Path(
    "../data/raw/mimic-iii/"
    "physionet.org/files/mimiciii-demo/1.4"
)

PROCESSED_DIR = Path("../data/processed/mimic3")

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("MIMIC-III:", MIMIC_DIR)
print("Processed:", PROCESSED_DIR)

MIMIC-III: ../data/raw/mimic-iii/physionet.org/files/mimiciii-demo/1.4
Processed: ../data/processed/mimic3


In [2]:
required_files = [
    "PATIENTS.csv",
    "ADMISSIONS.csv",
    "ICUSTAYS.csv",
    "DIAGNOSES_ICD.csv",
    "CHARTEVENTS.csv",
    "D_ITEMS.csv",
]

for file in required_files:
    path = MIMIC_DIR / file
    print(
        f"{file:20s}",
        "✓" if path.exists() else "✗"
    )

PATIENTS.csv         ✓
ADMISSIONS.csv       ✓
ICUSTAYS.csv         ✓
DIAGNOSES_ICD.csv    ✓
CHARTEVENTS.csv      ✓
D_ITEMS.csv          ✓


In [3]:
patients = pd.read_csv(
    MIMIC_DIR / "PATIENTS.csv"
)

admissions = pd.read_csv(
    MIMIC_DIR / "ADMISSIONS.csv"
)

icustays = pd.read_csv(
    MIMIC_DIR / "ICUSTAYS.csv"
)

diagnoses = pd.read_csv(
    MIMIC_DIR / "DIAGNOSES_ICD.csv"
)

d_items = pd.read_csv(
    MIMIC_DIR / "D_ITEMS.csv"
)

print("PATIENTS:", patients.shape)
print("ADMISSIONS:", admissions.shape)
print("ICUSTAYS:", icustays.shape)
print("DIAGNOSES:", diagnoses.shape)
print("D_ITEMS:", d_items.shape)

PATIENTS: (100, 8)
ADMISSIONS: (129, 19)
ICUSTAYS: (136, 12)
DIAGNOSES: (1761, 5)
D_ITEMS: (12487, 10)


In [ ]:
d_icd = pd.read_csv(
    MIMIC_DIR / "D_ICD_DIAGNOSES.csv"
)

print("D_ICD_DIAGNOSES:", d_icd.shape)
print(d_icd.columns.tolist())
display(d_icd.head())

D_ICD_DIAGNOSES: (14567, 4)
['row_id', 'icd9_code', 'short_title', 'long_title']


,row_id,icd9_code,short_title,long_title
0,1,01716,Erythem nod tb-oth test,Erythema nodosum with hypersensitivity reactio...
1,2,01720,TB periph lymph-unspec,"Tuberculosis of peripheral lymph nodes, unspec..."
2,3,01721,TB periph lymph-no exam,"Tuberculosis of peripheral lymph nodes, bacter..."
3,4,01722,TB periph lymph-exam unk,"Tuberculosis of peripheral lymph nodes, bacter..."
4,5,01723,TB periph lymph-micro dx,"Tuberculosis of peripheral lymph nodes, tuberc..."


In [6]:
# Load ICD-9 diagnosis dictionary
d_icd = pd.read_csv(
    MIMIC_DIR / "D_ICD_DIAGNOSES.csv"
)

# Join diagnosis codes with their descriptions
diagnoses_full = diagnoses.merge(
    d_icd[
        ["icd9_code", "short_title", "long_title"]
    ],
    on="icd9_code",
    how="left"
)

print("Joined diagnoses:", diagnoses_full.shape)

display(
    diagnoses_full.head()
)

Joined diagnoses: (1761, 7)


,row_id,subject_id,hadm_id,seq_num,icd9_code,short_title,long_title
0,112344,10006,142345,1,99591,Sepsis,Sepsis
1,112345,10006,142345,2,99662,React-oth vasc dev/graft,Infection and inflammatory reaction due to oth...
2,112346,10006,142345,3,5672,NaN,NaN
3,112347,10006,142345,4,40391,Hyp kid NOS w cr kid V,"Hypertensive chronic kidney disease, unspecifi..."
4,112348,10006,142345,5,42731,Atrial fibrillation,Atrial fibrillation


In [7]:
sepsis_dx = diagnoses_full[
    diagnoses_full["long_title"]
    .fillna("")
    .str.contains(
        "sepsis|septicemia|septic shock",
        case=False,
        na=False
    )
].copy()

print("Sepsis diagnosis rows:", len(sepsis_dx))
print(
    "Unique sepsis patients:",
    sepsis_dx["subject_id"].nunique()
)
print(
    "Unique sepsis admissions:",
    sepsis_dx["hadm_id"].nunique()
)

display(
    sepsis_dx[
        [
            "subject_id",
            "hadm_id",
            "icd9_code",
            "short_title",
            "long_title"
        ]
    ]
    .sort_values(["subject_id", "hadm_id"])
)

Sepsis diagnosis rows: 91
Unique sepsis patients: 26
Unique sepsis admissions: 38


,subject_id,hadm_id,icd9_code,short_title,long_title
0,10006,142345,99591,Sepsis,Sepsis
9,10006,142345,03819,Staphylcocc septicem NEC,Other staphylococcal septicemia
27,10013,165520,0389,Septicemia NOS,Unspecified septicemia
50,10019,177759,0389,Septicemia NOS,Unspecified septicemia
59,10019,177759,99592,Severe sepsis,Severe sepsis
...,...,...,...,...,...
1716,44212,163189,78552,Septic shock,Septic shock
1733,44212,163189,99592,Severe sepsis,Severe sepsis
1750,44228,103379,03842,E coli septicemia,Septicemia due to escherichia coli [E. coli]
1751,44228,103379,78552,Septic shock,Septic shock


In [8]:
sepsis_stays = icustays[
    icustays["subject_id"].isin(
        sepsis_dx["subject_id"]
    )
    &
    icustays["hadm_id"].isin(
        sepsis_dx["hadm_id"]
    )
].copy()

print("Sepsis ICU stays:", len(sepsis_stays))
print(
    "Unique sepsis ICU patients:",
    sepsis_stays["subject_id"].nunique()
)
print(
    "Unique sepsis ICU admissions:",
    sepsis_stays["hadm_id"].nunique()
)

display(
    sepsis_stays[
        [
            "subject_id",
            "hadm_id",
            "icustay_id",
            "first_careunit",
            "last_careunit",
            "intime",
            "outtime",
            "los"
        ]
    ].sort_values(["subject_id", "intime"])
)

Sepsis ICU stays: 41
Unique sepsis ICU patients: 26
Unique sepsis ICU admissions: 38


,subject_id,hadm_id,icustay_id,first_careunit,last_careunit,intime,outtime,los
0,10006,142345,206504,MICU,MICU,2164-10-23 21:10:15,2164-10-25 12:21:07,1.6325
2,10013,165520,264446,MICU,MICU,2125-10-04 23:38:00,2125-10-07 15:13:52,2.6499
4,10019,177759,228977,MICU,MICU,2163-05-14 20:43:56,2163-05-16 03:47:04,1.2938
7,10029,132349,226055,SICU,SICU,2139-09-23 12:37:10,2139-09-25 18:55:04,2.2624
11,10036,189483,227834,MICU,MICU,2185-03-24 16:57:05,2185-03-26 12:18:56,1.8068
12,10038,111115,235482,SICU,SICU,2144-02-11 12:10:34,2144-02-14 21:25:57,3.3857
17,10045,126949,203766,MICU,SICU,2129-11-24 22:46:57,2129-12-01 06:03:55,6.3034
19,10056,100375,285789,MICU,MICU,2129-05-02 00:12:39,2129-05-03 01:23:25,1.0491
21,10059,122098,248755,MICU,MICU,2150-08-22 17:33:50,2150-08-29 20:09:50,7.1083
22,10061,145203,223177,CCU,CCU,2107-01-16 11:34:20,2107-02-10 11:29:46,24.9968
